In [1]:
import os  # Lets us set JAVA_HOME before Spark starts.
import shutil  # Finds the Java command installed in Ubuntu.
from pathlib import Path  # Represents paths to Java and the project.

java_command = shutil.which("java")  # Find java on Ubuntu's PATH.
java_home = Path(java_command).resolve().parent.parent  # Find its installation folder.
os.environ["JAVA_HOME"] = str(java_home)  # Tell Spark which Java installation to use.

project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate the project from Ubuntu.

print("Java folder:", java_home)  # Confirm this is a Linux path.
print("Project found:", project_root.is_dir())  # Confirm the F: drive project is accessible.

Java folder: /usr/lib/jvm/java-17-openjdk-amd64
Project found: True


In [2]:
from pyspark.sql import SparkSession  # Import Spark's session builder.
from delta import configure_spark_with_delta_pip  # Add Java packages when Spark starts.

postgres_driver = "org.postgresql:postgresql:42.7.14"  # Choose the PostgreSQL JDBC driver version.

builder = (  # Define the Spark session's startup settings.
    SparkSession.builder
    .master("local[2]")  # Use two local processing threads.
    .appName("RetailBronzePostgres")  # Name this session.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL features.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # Enable Delta tables.
)

spark = configure_spark_with_delta_pip(  # Add Delta and the extra JDBC driver.
    builder, extra_packages=[postgres_driver]  # Download both Java packages at startup.
).getOrCreate()  # Start the configured Spark session.

print("Spark version:", spark.version)  # Confirm Spark started.
print("Java packages:", spark.sparkContext.getConf().get("spark.jars.packages"))  # Confirm both packages were requested.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 15:41:52 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 15:41:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9d587e1-945e-4c2f-8dca-2ddf80ca072f;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0

Spark version: 4.1.1
Java packages: io.delta:delta-spark_4.1_2.13:4.3.1,org.postgresql:postgresql:42.7.14


In [3]:
from dotenv import dotenv_values  # Read key-value settings from the local .env file.

db_settings = dotenv_values(project_root / ".env")  # Load the file without displaying its contents.
required_keys = ("POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_DB")  # Settings JDBC needs.
missing_keys = [key for key in required_keys if not db_settings.get(key)]  # Find absent or empty settings.

if missing_keys:  # Stop before connecting if the configuration is incomplete.
    raise ValueError(f"Missing database settings: {missing_keys}")  # Show only key names, never values.

jdbc_url = f"jdbc:postgresql://127.0.0.1:5433/{db_settings['POSTGRES_DB']}"  # Point to PostgreSQL from Ubuntu.
print("Database settings loaded:", True)  # Confirm success without showing credentials.

Database settings loaded: True


In [4]:
customers_df = (  # Define a DataFrame backed by the PostgreSQL table.
    spark.read.format("jdbc")  # Tell Spark to use its JDBC data source.
    .option("url", jdbc_url)  # Choose the database server and database.
    .option("dbtable", "public.customers")  # Choose the customers table.
    .option("user", db_settings["POSTGRES_USER"])  # Supply the database user privately.
    .option("password", db_settings["POSTGRES_PASSWORD"])  # Supply the password privately.
    .option("driver", "org.postgresql.Driver")  # Use the driver we loaded at Spark startup.
    .load()  # Create the DataFrame and obtain the table schema.
)

customers_df.printSchema()  # See how PostgreSQL types map to Spark types.
print("Customer rows:", customers_df.count())  # Check the expected 750 rows.
customers_df.select("customer_id", "city", "loyalty_tier").show(3, truncate=False)  # Inspect a small sample.

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- join_date: date (nullable = true)
 |-- active_flag: boolean (nullable = true)



Customer rows: 750
+-----------+-------+------------+
|customer_id|city   |loyalty_tier|
+-----------+-------+------------+
|C00001     |Lahore |Gold        |
|C00002     |Karachi|Silver      |
|C00003     |Karachi|Silver      |
+-----------+-------+------------+
only showing top 3 rows


In [5]:
bronze_customers_path = project_root / "lakehouse" / "bronze" / "data" / "customers"  # Choose the Delta table folder.
bronze_customers_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the Bronze data folder exists.

customers_df.write.format("delta").mode("overwrite").save(str(bronze_customers_path))  # Copy the PostgreSQL snapshot into Delta.

bronze_customers_df = spark.read.format("delta").load(str(bronze_customers_path))  # Read the saved table back.
print("Bronze customer rows:", bronze_customers_df.count())  # Check that 750 rows were saved.
print("Columns match source:", bronze_customers_df.columns == customers_df.columns)  # Compare column names and order.
print("Delta log exists:", (bronze_customers_path / "_delta_log").is_dir())  # Confirm this is a Delta table.

26/10/09 15:44:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Bronze customer rows: 750
Columns match source: True
Delta log exists: True


In [6]:
remaining_pg_tables = ("orders", "order_items", "payments")  # The database tables still to load.

for table_name in remaining_pg_tables:  # Inspect one PostgreSQL table at a time.
    source_df = (  # Define a DataFrame backed by that table.
        spark.read.format("jdbc")  # Read through JDBC.
        .option("url", jdbc_url)  # Connect to our PostgreSQL database.
        .option("dbtable", f"public.{table_name}")  # Choose the current table.
        .option("user", db_settings["POSTGRES_USER"])  # Use the local .env setting.
        .option("password", db_settings["POSTGRES_PASSWORD"])  # Use the password without printing it.
        .option("driver", "org.postgresql.Driver")  # Use the loaded PostgreSQL driver.
        .load()  # Obtain the table schema.
    )
    print(f"\n{table_name}: {source_df.count()} rows")  # Check the number of records.
    source_df.printSchema()  # Inspect the types supplied through JDBC.


orders: 4000 rows
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- promo_id: string (nullable = true)
 |-- subtotal: decimal(12,2) (nullable = true)
 |-- discount_amount: decimal(12,2) (nullable = true)
 |-- tax_amount: decimal(12,2) (nullable = true)
 |-- total_amount: decimal(12,2) (nullable = true)
 |-- currency_code: string (nullable = true)


order_items: 9868 rows
root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- line_total: decimal(12,2) (nullable = true)
 |-- currency_code: string (nullable = true)


payments: 4000 rows
root
 |-- payment_id: string (nullable 

In [7]:
for table_name in remaining_pg_tables:  # Process orders, order_items, then payments.
    source_df = (  # Read the current PostgreSQL table.
        spark.read.format("jdbc")  # Use Spark's JDBC reader.
        .option("url", jdbc_url)  # Connect to the database.
        .option("dbtable", f"public.{table_name}")  # Select this table.
        .option("user", db_settings["POSTGRES_USER"])  # Get the user from .env.
        .option("password", db_settings["POSTGRES_PASSWORD"])  # Get the password from .env.
        .option("driver", "org.postgresql.Driver")  # Use the PostgreSQL JDBC driver.
        .load()  # Obtain the source DataFrame.
    )
    source_count = source_df.count()  # Count the source rows.

    bronze_path = project_root / "lakehouse" / "bronze" / "data" / table_name  # Choose its Delta folder.
    bronze_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the parent folder exists.
    source_df.write.format("delta").mode("overwrite").save(str(bronze_path))  # Save a full snapshot.

    saved_df = spark.read.format("delta").load(str(bronze_path))  # Read the saved table.
    saved_count = saved_df.count()  # Count its rows.

    source_fields = [(field.name, field.dataType.simpleString()) for field in source_df.schema.fields]  # Record source names and types.
    saved_fields = [(field.name, field.dataType.simpleString()) for field in saved_df.schema.fields]  # Record saved names and types.
    schema_match = source_fields == saved_fields  # Check that names, order, and types match.

    print(f"{table_name}: source={source_count}, bronze={saved_count}, schema_match={schema_match}")  # Show validation.

    if saved_count != source_count or not schema_match:  # Detect an incomplete or changed copy.
        raise ValueError(f"Bronze validation failed for {table_name}")  # Stop and name the failing table.

orders: source=4000, bronze=4000, schema_match=True


order_items: source=9868, bronze=9868, schema_match=True


payments: source=4000, bronze=4000, schema_match=True
